In [0]:
%run /Workspace/Users/fayelatyr61@gmail.com/azure-databricks-realtime-health-platform/01-config

In [0]:
# Databricks notebook source
# MAGIC %run ./01-config

# COMMAND ----------

class SetupHelper:

    def __init__(self, catalog="dev"):

        conf = Config()

        self.catalog = catalog

        self.bronze = conf.bronze_schema
        self.silver = conf.silver_schema
        self.gold = conf.gold_schema

        # Sous-dossiers réellement utilisés par nos pipelines
        self.landing_zone = conf.base_dir_data + "/raw"
        self.checkpoint_base = conf.base_dir_checkpoint + "/checkpoints"

        self.initialized = False

    # --------------------------------------------------
    # CREATE MEDALLION SCHEMAS
    # --------------------------------------------------

    def create_schemas(self):

        print(f"Creating schemas in catalog {self.catalog}...")

        spark.sql(
            f"CREATE SCHEMA IF NOT EXISTS {self.catalog}.{self.bronze}"
        )

        spark.sql(
            f"CREATE SCHEMA IF NOT EXISTS {self.catalog}.{self.silver}"
        )

        spark.sql(
            f"CREATE SCHEMA IF NOT EXISTS {self.catalog}.{self.gold}"
        )

        self.initialized = True

        print("Done")

    # ==================================================
    # BRONZE
    # ==================================================

    def create_registered_users_bz(self):

        spark.sql(f"""
        CREATE TABLE IF NOT EXISTS
        {self.catalog}.{self.bronze}.registered_users_bz (
            user_id LONG,
            device_id LONG,
            mac_address STRING,
            registration_timestamp DOUBLE,
            load_time TIMESTAMP,
            source_file STRING
        )
        """)

    def create_gym_logins_bz(self):

        spark.sql(f"""
        CREATE TABLE IF NOT EXISTS
        {self.catalog}.{self.bronze}.gym_logins_bz (
            mac_address STRING,
            gym BIGINT,
            login DOUBLE,
            logout DOUBLE,
            load_time TIMESTAMP,
            source_file STRING
        )
        """)

    def create_kafka_multiplex_bz(self):

        spark.sql(f"""
        CREATE TABLE IF NOT EXISTS
        {self.catalog}.{self.bronze}.kafka_multiplex_bz (
            key STRING,
            value STRING,
            topic STRING,
            partition BIGINT,
            offset BIGINT,
            timestamp BIGINT,
            date DATE,
            week_part STRING,
            load_time TIMESTAMP,
            source_file STRING
        )
        PARTITIONED BY (topic, week_part)
        """)

    # ==================================================
    # SILVER
    # ==================================================

    def create_users(self):

        spark.sql(f"""
        CREATE TABLE IF NOT EXISTS
        {self.catalog}.{self.silver}.users (
            user_id BIGINT,
            device_id BIGINT,
            mac_address STRING,
            registration_timestamp TIMESTAMP
        )
        """)

    def create_gym_logs(self):

        spark.sql(f"""
        CREATE TABLE IF NOT EXISTS
        {self.catalog}.{self.silver}.gym_logs (
            mac_address STRING,
            gym BIGINT,
            login TIMESTAMP,
            logout TIMESTAMP
        )
        """)

    def create_user_profile(self):

        spark.sql(f"""
        CREATE TABLE IF NOT EXISTS
        {self.catalog}.{self.silver}.user_profile (
            user_id BIGINT,
            dob DATE,
            sex STRING,
            gender STRING,
            first_name STRING,
            last_name STRING,
            street_address STRING,
            city STRING,
            state STRING,
            zip INT,
            updated TIMESTAMP
        )
        """)

    def create_heart_rate(self):

        spark.sql(f"""
        CREATE TABLE IF NOT EXISTS
        {self.catalog}.{self.silver}.heart_rate (
            device_id LONG,
            time TIMESTAMP,
            heartrate DOUBLE,
            valid BOOLEAN
        )
        """)

    def create_user_bins(self):

        spark.sql(f"""
        CREATE TABLE IF NOT EXISTS
        {self.catalog}.{self.silver}.user_bins (
            user_id BIGINT,
            age STRING,
            gender STRING,
            city STRING,
            state STRING
        )
        """)

    def create_workouts(self):

        spark.sql(f"""
        CREATE TABLE IF NOT EXISTS
        {self.catalog}.{self.silver}.workouts (
            user_id INT,
            workout_id INT,
            time TIMESTAMP,
            action STRING,
            session_id INT
        )
        """)

    def create_completed_workouts(self):

        spark.sql(f"""
        CREATE TABLE IF NOT EXISTS
        {self.catalog}.{self.silver}.completed_workouts (
            user_id INT,
            workout_id INT,
            session_id INT,
            start_time TIMESTAMP,
            end_time TIMESTAMP
        )
        """)

    def create_workout_bpm(self):

        spark.sql(f"""
        CREATE TABLE IF NOT EXISTS
        {self.catalog}.{self.silver}.workout_bpm (
            user_id INT,
            workout_id INT,
            session_id INT,
            start_time TIMESTAMP,
            end_time TIMESTAMP,
            time TIMESTAMP,
            heartrate DOUBLE
        )
        """)

    def create_date_lookup(self):

        spark.sql(f"""
        CREATE TABLE IF NOT EXISTS
        {self.catalog}.{self.silver}.date_lookup (
            date DATE,
            week INT,
            year INT,
            month INT,
            dayofweek INT,
            dayofmonth INT,
            dayofyear INT,
            week_part STRING
        )
        """)

    # ==================================================
    # GOLD
    # ==================================================

    def create_workout_bpm_summary(self):

        spark.sql(f"""
        CREATE TABLE IF NOT EXISTS
        {self.catalog}.{self.gold}.workout_bpm_summary (
            workout_id INT,
            session_id INT,
            user_id BIGINT,
            age STRING,
            gender STRING,
            city STRING,
            state STRING,
            min_bpm DOUBLE,
            avg_bpm DOUBLE,
            max_bpm DOUBLE,
            num_recordings BIGINT
        )
        """)

    def create_gym_summary(self):

        spark.sql(f"""
        CREATE OR REPLACE VIEW
        {self.catalog}.{self.gold}.gym_summary AS

        SELECT
            TO_DATE(l.login) AS date,
            l.gym,
            l.mac_address,
            w.workout_id,
            w.session_id,

            ROUND(
                (CAST(l.logout AS LONG) -
                 CAST(l.login AS LONG)) / 60,
                2
            ) AS minutes_in_gym,

            ROUND(
                (CAST(w.end_time AS LONG) -
                 CAST(w.start_time AS LONG)) / 60,
                2
            ) AS minutes_exercising

        FROM {self.catalog}.{self.silver}.gym_logs l

        JOIN (

            SELECT
                u.mac_address,
                w.workout_id,
                w.session_id,
                w.start_time,
                w.end_time

            FROM
                {self.catalog}.{self.silver}.completed_workouts w

            INNER JOIN
                {self.catalog}.{self.silver}.users u

            ON w.user_id = u.user_id

        ) w

        ON l.mac_address = w.mac_address

        AND w.start_time
            BETWEEN l.login AND l.logout
        """)

    # ==================================================
    # GLOBAL SETUP
    # ==================================================

    def setup(self):

        import time

        start = int(time.time())

        print("\nStarting project setup...")

        self.create_schemas()

        # Bronze
        self.create_registered_users_bz()
        self.create_gym_logins_bz()
        self.create_kafka_multiplex_bz()

        # Silver
        self.create_users()
        self.create_gym_logs()
        self.create_user_profile()
        self.create_heart_rate()
        self.create_workouts()
        self.create_completed_workouts()
        self.create_workout_bpm()
        self.create_user_bins()
        self.create_date_lookup()

        # Gold
        self.create_workout_bpm_summary()
        self.create_gym_summary()

        print(
            f"Setup completed in "
            f"{int(time.time()) - start} seconds"
        )

    # ==================================================
    # VALIDATION
    # ==================================================

    def assert_table(self, schema, table):

        result = (
            spark.sql(
                f"SHOW TABLES IN "
                f"{self.catalog}.{schema}"
            )
            .filter(f"tableName = '{table}'")
            .count()
        )

        assert result == 1, (
            f"{self.catalog}.{schema}.{table} "
            f"is missing"
        )

        print(
            f"Found "
            f"{self.catalog}.{schema}.{table}: Success"
        )

    def validate(self):

        print("\nValidating project setup...")

        # Bronze
        self.assert_table(
            self.bronze,
            "registered_users_bz"
        )

        self.assert_table(
            self.bronze,
            "gym_logins_bz"
        )

        self.assert_table(
            self.bronze,
            "kafka_multiplex_bz"
        )

        # Silver
        for table in [
            "users",
            "gym_logs",
            "user_profile",
            "heart_rate",
            "workouts",
            "completed_workouts",
            "workout_bpm",
            "user_bins",
            "date_lookup"
        ]:
            self.assert_table(
                self.silver,
                table
            )

        # Gold
        self.assert_table(
            self.gold,
            "workout_bpm_summary"
        )

        self.assert_table(
            self.gold,
            "gym_summary"
        )

        print("\nAll objects validated successfully.")

    # ==================================================
    # CLEANUP
    # ==================================================

    def cleanup(self):

        print("\nCleaning project objects...")

        for schema in [
            self.gold,
            self.silver,
            self.bronze
        ]:

            spark.sql(
                f"DROP SCHEMA IF EXISTS "
                f"{self.catalog}.{schema} "
                f"CASCADE"
            )

        # Ne supprime que les dossiers créés
        # par nos pipelines, pas l'External Location entière.

        dbutils.fs.rm(
            self.landing_zone,
            True
        )

        dbutils.fs.rm(
            self.checkpoint_base,
            True
        )

        print("Cleanup completed.")

### Description de 02-setup

Ce notebook prépare l’environnement technique du projet grâce à une classe `SetupHelper`.

Il réutilise la configuration définie dans `01-config` afin de récupérer les chemins ADLS et les noms des couches Medallion sans les coder en dur.

La classe initialise les principaux paramètres du projet :

- `catalog` : catalogue Unity Catalog utilisé par l’environnement, par exemple `sbit_dev`.
- `bronze` : schéma contenant les données brutes.
- `silver` : schéma contenant les données nettoyées et transformées.
- `gold` : schéma contenant les données finales destinées au reporting.
- `landing_zone` : sous-répertoire utilisé pour les données entrantes.
- `checkpoint_base` : sous-répertoire utilisé pour les checkpoints des traitements streaming.

Le notebook contient ensuite les fonctions nécessaires pour :

- créer les schémas `bronze`, `silver` et `gold` ;
- créer les différentes tables du projet ;
- organiser les tables selon leur couche Medallion ;
- créer les tables et vues finales de reporting ;
- valider que tous les objets ont bien été créés ;
- nettoyer l’environnement lorsque l’on souhaite repartir de zéro.

La méthode `setup()` permet de lancer toute la création de l’environnement en une seule commande, tandis que `validate()` vérifie automatiquement que les tables attendues existent bien.

L’objectif est d’avoir un **script de setup automatisé, réutilisable et compatible avec plusieurs environnements**, tout en conservant une architecture claire :

`Bronze → Silver → Gold`